In [7]:
from playwright.sync_api import sync_playwright
import csv

URL = "https://www.amoremall.com/kr/ko/display/brand/detail?brandSn=27"  # Holitaul 예시

with sync_playwright() as p:
    browser = p.chromium.launch(headless=False)
    page = browser.new_page()
    page.goto(URL)
    page.wait_for_timeout(2000)  # 페이지 로딩 기다리기

Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.

In [8]:
from playwright.async_api import async_playwright
import csv

In [9]:
from playwright.async_api import async_playwright
import csv

URL = "https://www.amoremall.com/kr/ko/display/brand/detail?brandSn=27"  # Holitual

async def main():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)  # 창 보이게
        page = await browser.new_page()
        await page.goto(URL)
        await page.wait_for_timeout(3000)   # 페이지 로딩 잠깐 대기

        # (필요하면 스크롤 끝까지 내리기 – 상품이 더 로드되는 구조면)
        await page.evaluate("""
        () => new Promise(resolve => {
            let totalHeight = 0;
            const distance = 800;
            const timer = setInterval(() => {
                const scrollHeight = document.body.scrollHeight;
                window.scrollBy(0, distance);
                totalHeight += distance;
                if (totalHeight >= scrollHeight) {
                    clearInterval(timer);
                    resolve();
                }
            }, 200);
        })
        """)

        # 페이지 안에서 JS 실행해서 상품 정보 뽑기
        items = await page.evaluate("""
        () => {
            const cards = [...document.querySelectorAll('.productCard.shop-productCard.typeSmall')];
            const rows = [];

            for (const card of cards) {
                const body = card.querySelector('a.prodCardBody');
                const allText = card.textContent.replace(/\\s+/g, ' ').trim();

                const name =
                    (body && body.getAttribute('ap-prd-name')) ||
                    allText.split('  ')[0] || '';

                const priceMatch = allText.match(/([\\d,]+)원/);
                const discountMatch = allText.match(/(\\d+)%/);
                const ratingMatch = allText.match(/([0-5]\\.[0-9])\\s*\\(/);

                rows.push({
                    name,
                    price: priceMatch ? priceMatch[1] : '',
                    discount: discountMatch ? discountMatch[1] : '',
                    rating: ratingMatch ? ratingMatch[1] : '',
                });
            }

            return rows;
        }
        """)

        # CSV 저장 (상품명, 가격, 할인율, 평점)
        with open("홀리튜얼.csv", "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.writer(f)
            writer.writerow(["상품명", "가격", "할인율", "평점"])
            for item in items:
                writer.writerow([item["name"], item["price"], item["discount"], item["rating"]])

        await browser.close()

# Jupyter에서는 이렇게 그냥 await로 호출
await main()